In [1]:
import sys
!{sys.executable} -m pip install -r requirements.txt


!{sys.executable} -m pip install langchainhub==0.1.5


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import certifi
import requests
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import Tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
import requests


In [3]:
#from langchain.agents import create_react_agent
# use for reasoning and action
from langchain.prompts import ChatPromptTemplate

In [4]:
# load env variables
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

open_router_api_key = os.environ.get("open_router_api_key")
tavily_api_key = os.environ.get("tavily_api_key")

In [5]:
search_tool = TavilySearchResults(max_results=2, api_key=tavily_api_key)

In [6]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city."""
    api_key = os.environ.get("openweathermap_api_key")
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    response = requests.get(url)
    data = response.json()
    if data["cod"] != 200:
        return f"Error: {data['message']}"
    weather = data["weather"][0]["description"]
    temperature = data["main"]["temp"]
    return f"The current weather in {city} is {weather} with a temperature of {temperature}°C."

In [7]:
res=search_tool.invoke("give me about letest model on ai")
res

[{'url': 'https://www.evertune.ai/resources/ai-model-tracker',
  'content': "## Latest model from each provider\n\n Google: Gemini 3.8 Flash (Sep 2, 2026)\n Meta: Muse Spark 1.3 (Sep 2, 2026)\n Anthropic: Claude Fable 5.1 (Sep 1, 2026)\n DeepSeek: DeepSeek V4-Pro (0813 Official Release) (Aug 12, 2026)\n OpenAI: ChatGPT Work (Jul 9, 2026)\n\n## Frequently asked questions\n\nWhat is the most recent AI model release?\n:   The most recent tracked release is Google Gemini 3.8 Flash, released Sep 2, 2026. Google's newest everyday workhorse model, arriving about three weeks after 3.7 Flash. It is better at software engineering, agent workflows and multi-step reasoning than the version it replaces, at a lower cost than frontier models. Available through the Gemini API, AI Studio, Antigravity, Android Studio and Stitch. [...] What is the latest AI model from Google?\n:   The latest Google release in this tracker is Gemini 3.8 Flash, released Sep 2, 2026. Google's newest everyday workhorse model

In [8]:
llm = ChatOpenAI(
    model="openrouter/free",
    openai_api_key=os.getenv("open_router_api_key"),
    openai_api_base="https://openrouter.ai/api/v1",
)


In [9]:
response=llm.invoke("tell me a joke")
response


AIMessage(content="Here's a light-hearted one for you:\n\n**Why don’t scientists trust atoms anymore?**  \n*Because they make up everything!* 😄  \n\n*(Bonus groaner: I’m reading a book about anti-gravity. It’s impossible to put down!)*  \n\nHope that brought a smile! Let me know if you’d like another — punny, dad-joke style, or something else entirely. 😊", response_metadata={'token_usage': {'completion_tokens': 292, 'prompt_tokens': 21, 'total_tokens': 313, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 209, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_name': 'openrouter/free', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, 

In [10]:

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful AI assistant.

Use the available tools when you need current information.
Do not invent tools that are not available.
"""
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
prompt

ChatPromptTemplate(input_variables=['input'], input_types={'agent_scratchpad': typing.List[typing.Union[langchain_core.messages.ai.AIMessage, langchain_core.messages.human.HumanMessage, langchain_core.messages.chat.ChatMessage, langchain_core.messages.system.SystemMessage, langchain_core.messages.function.FunctionMessage, langchain_core.messages.tool.ToolMessage]]}, partial_variables={'agent_scratchpad': []}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], template='You are a helpful AI assistant.\n\nUse the available tools when you need current information.\nDo not invent tools that are not available.\n')), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], template='{input}')), MessagesPlaceholder(variable_name='agent_scratchpad', optional=True)])

In [11]:
tools = [search_tool,get_weather]

In [13]:
#create Agent
agent = create_openai_tools_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [14]:
#executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [15]:
#Run
response = agent_executor.invoke({
    "input": (
    "tell me the latest news about usa"
    "finds it's current weather in new york"
    )
    })
response



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'usafinds latest news'}`


[{'url': 'https://www.cbsnews.com/tag/usaid', 'content': 'USAID to hire lawyers to investigate employees who talked to the press, alleged job posting shows USAID cuts impact Myanmar earthquake victims'}, {'url': 'https://www.af.mil/News', 'content': '# Readout of USAF Chief of Staff Gen. Wilsbach’s counterpart visit with Bulgarian AF Commander Maj. Gen. Nikolay Rusev\n\nBilateral talks at the Pentagon focused on NATO burden-sharing, eastern flank collective deterrence and Bulgaria’s acquisition of eight F-16 Block 70 fighter aircraft.\n\n# Hernandez-Baquero swears in as Department of the Air Force space acquisition and integration lead\n\nDr. Erich "HB" Hernandez-Baquero was sworn in as the second Assistant Secretary of the Air Force for Space Acquisition and Integration during a Pentagon ceremony this morning, following his confirmation to the role by the U.S. Sen

{'input': "tell me the latest news about usafinds it's current weather in new york",
 'output': 'Here\'s the latest news from the United States Air Force (USAF) and the current weather in New York:\n\n## 📰 Latest USAF News (from AF.mil)\nBased on recent updates from the official U.S. Air Force news site:\n\n1. **Leadership & International Engagement**\n   - USAF Chief of Staff Gen. David Wilsbach met with Bulgarian AF Commander Maj. Gen. Nikolay Rusev at the Pentagon to discuss NATO burden-sharing, eastern flank collective deterrence, and Bulgaria\'s planned acquisition of eight F-16 Block 70 fighter aircraft.\n   - Dr. Erich "HB" Hernandez-Baquero was sworn in as the Assistant Secretary of the Air Force for Space Acquisition and Integration following Senate confirmation.\n\n2. **Modernization & Procurement**\n   - The Air Force awarded a production contract for the next-generation Aerospace Readiness Enterprise System (ARES), aimed at improving logistics and readiness tracking.\n   - 